**1. Obtención de Datos**

Este script recolecta datos de videos de YouTube utilizando la API oficial, a partir de un conjunto de palabras clave organizadas por áreas del conocimiento. Para cada video obtiene información como título, canal, fecha de publicación, vistas, likes, comentarios, duración y suscriptores del canal.

El proceso filtra los videos por fecha mínima, evita duplicados y guarda avances parciales para no perder información en caso de interrupciones. Finalmente, genera un archivo CSV por cada área y un archivo consolidado con todos los datos recolectados.

In [ ]:
import pandas as pd
from googleapiclient.discovery import build
from datetime import datetime, timezone
from time import sleep
import isodate

# CONFIGURACIÓN
API_KEY = "API_KEY"
MAX_VIDEOS = 500
FECHA_MINIMA = datetime(2020, 1, 1, tzinfo=timezone.utc)

# DICCIONARIO DE ÁREAS
diccionario_areas = {
    "fisica": [
        "física cuántica", "relatividad", "mecánica clásica", "experimentos de física",
        "universo y materia", "física nuclear", "termodinámica", "ondas y acústica",
        "electromagnetismo", "física teórica", "gravedad cuántica", "campo electromagnético",
        "ciencia del espacio", "teoría de cuerdas", "agujeros negros",
        "colisionadores de partículas", "modelos del universo", "ciencia de partículas",
        "física aplicada", "física experimental", "ciencia del tiempo", "teoría del caos",
        "interacciones fundamentales", "mecánica cuántica", "ondas gravitacionales"
    ],

    "quimica": [
        "química orgánica", "reacciones químicas", "bioquímica", "química inorgánica",
        "catálisis", "polímeros", "química analítica", "termocatálisis", "electroquímica",
        "física química", "tabla periódica", "ácidos y bases", "química experimental",
        "química medicinal", "química ambiental", "compuestos químicos",
        "ciencia de los materiales", "reacciones ácido-base", "estructuras moleculares",
        "química de alimentos", "nanomateriales", "técnicas espectroscópicas",
        "equilibrio químico", "laboratorio de química", "organoquímica"
    ],

    "biologia": [
        "genética", "biología molecular", "ecología", "microbiología", "evolución",
        "biotecnología", "zoología", "botánica", "biología celular", "fisiología",
        "ADN y ARN", "mutaciones genéticas", "biología marina", "sistemas ecológicos",
        "inmunología", "biología del desarrollo", "células madre",
        "organismos microscópicos", "ecología urbana", "biología vegetal",
        "interacciones biológicas", "biología de la conservación",
        "biología de sistemas", "biología humana", "diversidad genética"
    ],

    "matematicas": [
        "álgebra", "cálculo diferencial", "estadística", "geometría",
        "análisis matemático", "teoría de números", "probabilidad",
        "matemáticas aplicadas", "topología", "matemáticas discretas",
        "ecuaciones diferenciales", "matemáticas financieras",
        "matemáticas en física", "modelado matemático", "optimización",
        "números complejos", "matemáticas y ciencia", "cálculo integral",
        "aritmética", "sistemas dinámicos", "teoría de grafos",
        "álgebra lineal", "transformadas matemáticas", "análisis de datos",
        "teoremas matemáticos"
    ],

    "ingenieria": [
        "ingeniería civil", "ingeniería eléctrica", "ingeniería mecánica",
        "ingeniería electrónica", "ingeniería química", "automatización",
        "robótica", "diseño estructural", "energías renovables",
        "procesos industriales", "ingeniería ambiental", "circuitos electrónicos",
        "mecatrónica", "ingeniería de software", "diseño mecánico",
        "sistemas térmicos", "control automático", "instrumentación",
        "construcción inteligente", "estructuras metálicas",
        "termodinámica aplicada", "ingeniería de materiales",
        "transporte y logística", "mantenimiento industrial",
        "ciencia e ingeniería"
    ],

    "ciencias_sociales": [
        "sociología", "antropología", "psicología social", "economía",
        "ciencia política", "educación", "derechos humanos",
        "historia social", "comunicación", "trabajo social",
        "movimientos sociales", "desigualdad", "ciudadanía",
        "democracia", "relaciones internacionales",
        "sociología de la educación", "género y sociedad",
        "historia contemporánea", "interacción social",
        "política pública", "antropología cultural",
        "cambio social", "teoría social",
        "sistemas económicos", "sociedad y tecnología"
    ],

    "salud_medicina": [
        "medicina general", "enfermería", "salud pública",
        "epidemiología", "farmacología", "nutrición",
        "biomedicina", "cardiología", "oncología",
        "neurociencia", "células madre", "vacunas",
        "sistema inmunológico", "virus y bacterias",
        "cuidados intensivos", "salud mental",
        "medicina preventiva", "diagnóstico médico",
        "cáncer", "diabetes", "primeros auxilios",
        "biología médica", "cirugía", "medicina interna",
        "terapias alternativas"
    ],

    "tecnologia_ia": [
        "inteligencia artificial", "aprendizaje automático",
        "redes neuronales", "procesamiento de lenguaje natural",
        "robótica", "visión por computadora", "big data",
        "internet de las cosas", "computación en la nube",
        "ciberseguridad", "algoritmos", "modelos predictivos",
        "tecnología educativa", "chatbots", "análisis de datos",
        "blockchain", "sistemas inteligentes", "ética en IA",
        "tecnología emergente", "machine learning en medicina",
        "automatización industrial", "programación con IA",
        "modelos generativos", "transformación digital",
        "hardware de IA"
    ]
}

# FUNCIONES
def buscar_videos_youtube(query, youtube, max_results=50):
    return youtube.search().list(
        q=query,
        type="video",
        part="snippet",
        maxResults=max_results,
        regionCode="MX",
        relevanceLanguage="es",
        order="viewCount"
    ).execute().get("items", [])


def obtener_estadisticas_video(youtube, video_id):
    response = youtube.videos().list(
        part="statistics,contentDetails,status",
        id=video_id
    ).execute()

    items = response.get("items", [])
    if not items:
        return {"viewCount": None, "likeCount": None, "commentCount": None, "duration": None, "madeForKids": None}

    item = items[0]
    stats = item.get("statistics", {})
    content = item.get("contentDetails", {})
    status = item.get("status", {})

    try:
        duration = isodate.parse_duration(content.get("duration")).total_seconds()
    except:
        duration = None

    return {
        "viewCount": int(stats.get("viewCount", 0)),
        "likeCount": int(stats.get("likeCount", 0)),
        "commentCount": int(stats.get("commentCount", 0)),
        "duration": duration,
        "madeForKids": status.get("madeForKids", None)
    }


def obtener_datos_canal(youtube, channel_id):
    try:
        response = youtube.channels().list(part="statistics", id=channel_id).execute()
        items = response.get("items", [])
        if not items:
            return {"subscriberCount": None}

        stats = items[0].get("statistics", {})
        return {
            "subscriberCount": int(stats.get("subscriberCount", 0)) if not stats.get("hiddenSubscriberCount", False) else None
        }
    except:
        return {"subscriberCount": None}


# FUNCIÓN PRINCIPAL
def recolectar_videos_area(area, palabras_clave):
    youtube = build("youtube", "v3", developerKey=API_KEY)
    videos_totales = []
    ids_existentes = set()

    for palabra in palabras_clave:
        try:
            items = buscar_videos_youtube(palabra, youtube)
        except:
            continue

        for item in items:
            try:
                video_id = item["id"]["videoId"]

                if video_id in ids_existentes:
                    continue

                fecha_publicacion = datetime.fromisoformat(
                    item["snippet"]["publishedAt"].replace("Z", "+00:00")
                )

                if fecha_publicacion < FECHA_MINIMA:
                    continue

                stats = obtener_estadisticas_video(youtube, video_id)
                canal_stats = obtener_datos_canal(youtube, item["snippet"]["channelId"])

                videos_totales.append({
                    "videoId": video_id,
                    "titulo": item["snippet"]["title"],
                    "canal": item["snippet"]["channelTitle"],
                    "canal_id": item["snippet"]["channelId"],
                    "fecha_publicacion": item["snippet"]["publishedAt"],
                    "descripcion": item["snippet"]["description"],
                    "palabra_clave": palabra,
                    "vistas": stats["viewCount"],
                    "likes": stats["likeCount"],
                    "comentarios": stats["commentCount"],
                    "duracion_segundos": stats["duration"],
                    "es_para_ninos": stats["madeForKids"],
                    "suscriptores_canal": canal_stats["subscriberCount"]
                })

                ids_existentes.add(video_id)

                # guardado parcial cada 50
                if len(videos_totales) % 50 == 0:
                    df_temp = pd.DataFrame(videos_totales)
                    df_temp.to_csv(f"videos_{area}_parcial.csv", index=False)

                if len(ids_existentes) >= MAX_VIDEOS:
                    break

            except:
                print(f"Cuota alcanzada en {area}. Guardando datos...")

                df = pd.DataFrame(videos_totales)
                df.drop_duplicates(subset="videoId", inplace=True)
                df.to_csv(f"videos_{area}.csv", index=False)

                return df

        if len(ids_existentes) >= MAX_VIDEOS:
            break

        sleep(1)

    df = pd.DataFrame(videos_totales)
    df.drop_duplicates(subset="videoId", inplace=True)
    df.to_csv(f"videos_{area}.csv", index=False)

    print(f"Finalizado: videos_{area}.csv")
    return df


# EJECUCIÓN PARA TODAS LAS ÁREAS
dfs = []

for area, palabras in diccionario_areas.items():
    print(f"Procesando: {area}")
    df_area = recolectar_videos_area(area, palabras)

    if df_area is not None and not df_area.empty:
        df_area["area"] = area
        dfs.append(df_area)

# UNIR TODO
if dfs:
    df_final = pd.concat(dfs, ignore_index=True)
    df_final.to_csv("videos_todas_areas.csv", index=False)
    print("Archivo final generado: videos_todas_areas.csv")
else:
    print("No se generaron datos.")

**1.1 Limpieza de Datos**

Este script realiza la limpieza de los archivos CSV previamente recolectados. Elimina registros con valores faltantes en columnas clave, detecta el idioma de los títulos y filtra únicamente aquellos en español.

Además, descarta videos con métricas inválidas (sin vistas, likes o comentarios) para asegurar la calidad de los datos. Finalmente, guarda una versión limpia de cada archivo en una nueva carpeta, lista para su análisis.


In [ ]:
import pandas as pd
from langdetect import detect, LangDetectException
import os

# 1. Carpetas
directorio_csv = "datos_originales"
directorio_salida = "datos_limpios"
os.makedirs(directorio_salida, exist_ok=True)

# 2. Columnas clave para limpiar
columnas_obligatorias = ["titulo", "descripcion", "vistas", "likes", "comentarios"]

# 3. Función para detectar idioma
def detectar_idioma(texto):
    try:
        return detect(texto)
    except LangDetectException:
        return "unknown"

for archivo in os.listdir(directorio_csv):
    if archivo.endswith(".csv"):
        ruta_entrada = os.path.join(directorio_csv, archivo)
        df = pd.read_csv(ruta_entrada)

        # 4. Eliminar filas con valores vacíos en columnas clave
        df = df.dropna(subset=columnas_obligatorias)

        # Detectar idioma y filtrar español
        df["idioma"] = df["titulo"].apply(detectar_idioma)
        df = df[df["idioma"] == "es"]

        # Filtrar videos con estadísticas relevantes
        df = df[(df["vistas"] > 0) & (df["likes"] > 0) & (df["comentarios"] >= 0)]

        # 5. Guardar archivo limpio
        nombre_salida = archivo.replace(".csv", "_limpio.csv")
        ruta_salida = os.path.join(directorio_salida, nombre_salida)
        df.to_csv(ruta_salida, index=False)

        print(f"Archivo limpio creado: {ruta_salida}")

print("Limpieza completada para todos los archivos.")
